In [8]:
import time
import warnings
from collections import Counter
from itertools import combinations

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    Ridge,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MinMaxScaler,
    OrdinalEncoder,
    PolynomialFeatures,
)
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
warnings.filterwarnings("ignore")

In [3]:
train_processed = pd.read_csv("train_processed.csv")
val_processed = pd.read_csv("val_processed.csv")

categorical_features = ["shopId", "itemId", "modelId", "cat_id", "brand", "promotionId"]
price_features = ['priceBeforeDiscount', 'item_price_min', 'item_price_max']

# Drop categorical features
train_processed = train_processed.drop(columns=categorical_features)
val_processed = val_processed.drop(columns=categorical_features)

# Log transform price features
train_processed[price_features] = np.log1p(train_processed[price_features])
val_processed[price_features] = np.log1p(val_processed[price_features])

# Feature (X)
X_train = train_processed.drop(columns=["price"])
X_val = val_processed.drop(columns=["price"])

# Target (y)
y_train = train_processed["price"]
y_val = val_processed["price"]

## PCA

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)

print(f"Original features : {X_train.shape[1]}")
print(f"PCA features      : {X_train_pca.shape[1]}")
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.4f}")

Original features : 17
PCA features      : 13
Explained variance: 0.9653


In [6]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train_pca, y_train_log)

y_pred_log = model.predict(X_val_pca)
y_pred = np.expm1(y_pred_log)

In [7]:
# Metrics
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mape = mean_absolute_percentage_error(y_val, y_pred) * 100  

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE : 45.6132
RMSE: 128.3161
MAPE: 15.29%


### SVD

In [15]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

svd = TruncatedSVD(n_components=14, random_state=42)

X_train_svd = svd.fit_transform(X_train_scaled)
X_val_svd = svd.transform(X_val_scaled)

print(f"Original features : {X_train.shape[1]}")
print(f"SVD features      : {X_train_svd.shape[1]}")
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.4f}")

Original features : 17
SVD features      : 14
Explained variance: 0.9910


In [16]:
y_train_log = np.log1p(y_train)

model = LinearRegression()
model.fit(X_train_svd, y_train_log)

y_pred_log = model.predict(X_val_svd)
y_pred = np.expm1(y_pred_log)

In [17]:
# Metrics
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mape = mean_absolute_percentage_error(y_val, y_pred) * 100  

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE : 44.8346
RMSE: 126.3060
MAPE: 15.18%
